In [ ]:
import os

BASE = "/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360"

TRAIN_DIR = os.path.join(BASE, "Training")
TEST_DIR  = os.path.join(BASE, "Test")   

assert os.path.isdir(TRAIN_DIR), f"Not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"Not found: {TEST_DIR}"

train_classes = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])

test_classes = sorted([
    d for d in os.listdir(TEST_DIR)
    if os.path.isdir(os.path.join(TEST_DIR, d))
])

NUM_CLASSES = len(train_classes)

print(f"Training dir    : {TRAIN_DIR}")
print(f"Test dir        : {TEST_DIR}")
print(f"Train classes   : {len(train_classes)}")
print(f"Test  classes   : {len(test_classes)}")
print(f"First 5 classes : {train_classes[:5]}")
print(f"Last  5 classes : {train_classes[-5:]}")

total_train = sum(
    len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    for cls in train_classes
)

total_test = sum(
    len(os.listdir(os.path.join(TEST_DIR, cls)))
    for cls in test_classes
)

print(f"Total train images : {total_train}")
print(f"Total test  images : {total_test}")

In [ ]:
import shutil
import os

CODE_SRC = "/kaggle/input/datasets/mayankkcd/mycodee/EEA Project"   
CODE_DST = "/kaggle/working"

folders = ["data", "models", "utils", "compression"]

for folder in folders:
    src = os.path.join(CODE_SRC, folder)
    dst = os.path.join(CODE_DST, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied {folder}/")
    else:
        print(f"WARNING: {src} not found — check your code dataset name")

for fname in ["config.py", "main.py"]:
    src = os.path.join(CODE_SRC, fname)
    dst = os.path.join(CODE_DST, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {fname}")

os.makedirs("/kaggle/working/compressed_models", exist_ok=True)
print("\nDone. Files ready in /kaggle/working/")

In [ ]:
for root, dirs, files in os.walk("/kaggle/working"):
    print("Folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files)
    print("-" * 40)

In [ ]:

import subprocess
subprocess.run(["pip", "install", "scikit-learn", "-q"])

import sys
sys.path.insert(0, "/kaggle/working")

import torch
import numpy as np
import cv2

from config          import config_device
from data.data_loader import get_dataloader
from models.mnist    import mnist_model
from utils.training  import train_and_eval, evaluate
from compression.pruning import prune_model, quantize_model
from utils.loading   import save_model_npz, load_model_from_npz

print("All imports OK")
print(f"PyTorch  : {torch.__version__}")
print(f"OpenCV   : {cv2.__version__}")
print(f"NumPy    : {np.__version__}")

device = config_device()
print(f"Device   : {device}")

In [ ]:
BATCH_SIZE  = 64
IMAGE_SIZE  = (100, 100)   # Mihai Oltean fruits-360 native size
NUM_WORKERS = 2

train_loader, train_ds = get_dataloader(
    root_dir    = TRAIN_DIR,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    image_size  = IMAGE_SIZE,
    pin_memory  = torch.cuda.is_available(),
)

test_loader, test_ds = get_dataloader(
    root_dir    = TEST_DIR,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    image_size  = IMAGE_SIZE,
    pin_memory  = torch.cuda.is_available(),
)


print(f"Train samples  : {len(train_ds):,}")
print(f"Test  samples  : {len(test_ds):,}")
print(f"Train batches  : {len(train_loader)}")
print(f"Test  batches  : {len(test_loader)}")
print(f"Num classes    : {train_ds.num_classes}")

batch = next(iter(train_loader))
print(f"\nBatch keys     : {list(batch.keys())}")
print(f"image shape    : {batch['image'].shape}")    # (64, 3, 100, 100)
print(f"label shape    : {batch['label'].shape}")    # (64,)
print(f"features shape : {batch['features'].shape}") # (64, 12)
print(f"label range    : {batch['label'].min()} - {batch['label'].max()}")

In [ ]:
NUM_CLASSES = train_ds.num_classes
INPUT_SIZE  = 100

model = mnist_model(
    num_classes = NUM_CLASSES,
    input_size  = INPUT_SIZE,
).to(device)

# verify model
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model           : SmallCIFARNet")
print(f"Num classes     : {NUM_CLASSES}")
print(f"Input size      : {INPUT_SIZE}x{INPUT_SIZE}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size      : {total_params * 4 / 1024 / 1024:.2f} MB  (float32)")
print(f"\n{model}")

In [ ]:
print("=" * 60)
print("STAGE 0 : BASELINE TRAINING")
print("=" * 60)

train_and_eval(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    device       = device,
    epochs       = 8,       # 30 epochs gives reasonable accuracy
    lr           = 1e-3,
    weight_decay = 1e-4,
)

In [ ]:
print("Evaluating baseline model (no compression)...")

criterion_baseline = torch.nn.CrossEntropyLoss()
baseline_results = evaluate(model, test_loader, criterion_baseline, device)

baseline_top1 = baseline_results['accuracy']
baseline_top5 = baseline_results['top5_accuracy']

print(f"Baseline Top-1 Accuracy : {baseline_top1:.2f}%")
print(f"Baseline Top-5 Accuracy : {baseline_top5:.2f}%")

In [ ]:
print("=" * 60)
print("STAGE 1 : L1 PRUNING  (90%)")
print("=" * 60)

prune_model(model, 0.90)

train_and_eval(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    device       = device,
    epochs       = 5,
    lr           = 1e-4,    
    weight_decay = 0.0,
)

def count_mask_sparsity(model):
    total = zeros = 0
    for m in model.modules():
        if hasattr(m, "mask"):
            total += m.mask.numel()
            zeros += (m.mask == 0).sum().item()
    if total > 0:
        print(f"Masked sparsity : {100.0 * zeros / total:.2f}%")
    else:
        print("No masks found.")

count_mask_sparsity(model)


PTH_PATH = "/kaggle/working/compressed_models/model.pth"
torch.save(model.state_dict(), PTH_PATH)
print(f"Saved → {PTH_PATH}  "
      f"({os.path.getsize(PTH_PATH) / 1024:.1f} KB)")

In [ ]:
print("=" * 60)
print("STAGE 2 : K-MEANS QUANTIZATION  (K=16)")
print("=" * 60)

quantize_model(model, 16)

train_and_eval(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    device       = device,
    epochs       = 3,
    lr           = 1e-4,
    weight_decay = 0.0,
)

In [ ]:
print("=" * 60)
print("STAGE 3 : HUFFMAN CODING")
print("=" * 60)

NPZ_PATH = "/kaggle/working/compressed_models/compressed.npz"
save_model_npz(model, NPZ_PATH)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

model2 = mnist_model(
    num_classes = NUM_CLASSES,
    input_size  = INPUT_SIZE,
)

model2 = load_model_from_npz(model2, NPZ_PATH, device)

print("\n── Compressed model evaluation ──────────────────────")
results = evaluate(model2, test_loader, criterion, device)
print(results)

In [ ]:
import tracemalloc

tracemalloc.start()

model_for_mem = mnist_model(num_classes=NUM_CLASSES, input_size=INPUT_SIZE)
model_for_mem = load_model_from_npz(model_for_mem, NPZ_PATH, device)

current_ram, peak_ram = tracemalloc.get_traced_memory()
tracemalloc.stop()

cpu_ram_mb = peak_ram / (1024 ** 2)

def model_memory_mb(m):
    total = sum(
        p.numel() * p.element_size()
        for p in list(m.parameters()) + list(m.buffers())
    )
    return total / (1024 ** 2)

in_memory_mb = model_memory_mb(model2)

if torch.cuda.is_available():
    gpu_allocated = torch.cuda.memory_allocated(device) / (1024 ** 2)
    gpu_reserved  = torch.cuda.memory_reserved(device)  / (1024 ** 2)
else:
    gpu_allocated = 0
    gpu_reserved  = 0

pth_size_kb = os.path.getsize(PTH_PATH) / 1024
npz_size_kb = os.path.getsize(NPZ_PATH) / 1024
ratio       = pth_size_kb / npz_size_kb

print("\n" + "=" * 60)
print("   EXPERIMENTAL RESULTS SUMMARY")
print("=" * 60)

print("\n--- ACCURACY ---")
print(f"  Before compression (baseline) Top-1 : {baseline_top1:.2f}%")
print(f"  Before compression (baseline) Top-5 : {baseline_top5:.2f}%")
print(f"  After  compression            Top-1 : {results['accuracy']:.2f}%")
print(f"  After  compression            Top-5 : {results['top5_accuracy']:.2f}%")
print(f"  Accuracy drop Top-1                 : "
      f"{baseline_top1 - results['accuracy']:.2f}%")
print(f"  Accuracy drop Top-5                 : "
      f"{baseline_top5 - results['top5_accuracy']:.2f}%")

print("\n--- STORAGE MEMORY ---")
print(f"  Before compression (model.pth) : {pth_size_kb:.1f} KB "
      f"({pth_size_kb/1024:.2f} MB)")
print(f"  After  compression (.npz)      : {npz_size_kb:.1f} KB "
      f"({npz_size_kb/1024:.2f} MB)")

print("\n--- COMPRESSION RATIO ---")
print(f"  Ratio         : {ratio:.2f}x")
print(f"  Space saved   : {(1 - 1/ratio)*100:.1f}%")

print("\n--- RUNTIME MEMORY AFTER COMPRESSION ---")
print(f"  Model in-memory (float32) : {in_memory_mb:.2f} MB")
print(f"  Peak CPU RAM (load time)  : {cpu_ram_mb:.2f} MB")
if torch.cuda.is_available():
    print(f"  GPU allocated             : {gpu_allocated:.2f} MB")
    print(f"  GPU reserved              : {gpu_reserved:.2f} MB")

print("=" * 60)


print("\nOutput files:")
for f in sorted(os.listdir("/kaggle/working/compressed_models")):
    p = f"/kaggle/working/compressed_models/{f}"
    print(f"  {f:<30} {os.path.getsize(p)/1024:.1f} KB")